In [1]:
from azure.storage.blob import BlobServiceClient
from io import BytesIO
import pandas as pd
import os
from dotenv import load_dotenv

In [2]:
def blob_to_df(container_client, blob_name):
    client = container_client.get_blob_client(blob_name)
    data = client.download_blob().readall()
    stream = BytesIO(data)
    df = pd.read_csv(stream)
    return df

In [3]:
load_dotenv()

True

In [18]:
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
container_name = "recsys"

blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service_client.get_container_client(container_name)

In [ ]:
user_map_df = pd.DataFrame({
    "name": ['Qiqi','Torid', 'Uche'],
    'user_id': ['U19519', 'U12188', 'U22123']
})
user_map_df

In [ ]:
user_map = "user_map/user_map.csv"

container_client.get_blob_client(user_map).upload_blob(
        user_map_df.to_csv(index=False), overwrite=True)

In [44]:
news = pd.read_csv('news.tsv', header=None, sep='\t') #news.tsv is deleted and just uploaded from local
news.columns = [
    'content_id',
    "category",
    "sub_category",
    "title",
    "abstract",
    "URL",
    "title_entities",
    "abstract_entities"]

news = news.drop(columns=['URL', 'title_entities', 'abstract_entities'])

In [45]:
import re
import wordninja

def format_category(s):
    s = wordninja.split(s)
    s = " ".join(s)
    s = s.title()
    
    if s == 'Foodanddrink':
        s = 'Food and Drink'
    if s == 'Tv':
        s = 'TV'
    return s

def format_subcategory(s):
    s = wordninja.split(s)
    s = " ".join(s)
    s = s.title()

    s = s.replace('Tv', 'TV')
    s = s.replace('Ce Leb', 'Celeb')
    s = s.replace('Diy', 'DIY')
    s = s.replace('Us', 'US')
    s = s.replace('Nfl', 'NFL')
    s = s.replace('Mma', 'MMA')

    if s == 'Icehockey Nhl':
        s = 'Ice Hockey NHL'

    return s

In [46]:
news['category'] = news['category'].apply(format_category)
news['sub_category'] = news['sub_category'].apply(format_subcategory)

In [48]:
news = news.dropna()
news.isnull().sum().any()

False

In [49]:
raw_content = 'content/content.csv'
container_client.get_blob_client(raw_content).upload_blob(
        news.to_csv(index=False), overwrite=True)

{'etag': '"0x8DD92E0809EFC59"',
 'last_modified': datetime.datetime(2025, 5, 14, 12, 11, 48, tzinfo=datetime.timezone.utc),
 'content_md5': bytearray(b'\x0c\xdfAh\xeb\xaf\x98\x7f\x8a\xe0\xa3\xce\x94k5m'),
 'client_request_id': '9c375af2-30bc-11f0-8759-7c1e522d67b3',
 'request_id': '29e45481-301e-0003-29c9-c4614a000000',
 'version': '2025-05-05',
 'version_id': None,
 'date': datetime.datetime(2025, 5, 14, 12, 11, 47, tzinfo=datetime.timezone.utc),
 'request_server_encrypted': True,
 'encryption_key_sha256': None,
 'encryption_scope': None,
 'structured_body': None}

In [54]:
raw_content = 'content/content.csv'
raw_content_df = blob_to_df(container_client, raw_content)
raw_content_df.isnull().sum().any()
len(raw_content_df['content_id'].unique())

48616

In [56]:
raw_interaction = 'interactions/interactions.csv'
raw_interaction_df = blob_to_df(container_client, raw_interaction)
raw_interaction_df.isnull().sum().any()
len(raw_interaction_df['content_id'].unique())

920

In [57]:
processed_interaction = 'processed_interactions/processed_interactions.csv'
processed_interaction_df = blob_to_df(container_client, processed_interaction)
processed_interaction_df.isnull().sum().any()
len(processed_interaction_df['content_id'].unique())

920

In [65]:
all_recommendations = 'all_recommendations/all_recommendations.csv'
all_recommendations_df = blob_to_df(container_client, all_recommendations)
all_recommendations_df.isnull().sum().any()
len(all_recommendations_df)

100

In [64]:
merged_recommendations = 'merged_recommendations/merged_recommendations.csv'
merged_recommendations_df = blob_to_df(container_client, merged_recommendations)
merged_recommendations_df.isnull().sum().any()
len(merged_recommendations_df['content_id'].unique())

559

In [67]:
lists = [list(merged_recommendations_df['content_id']), list(processed_interaction_df['content_id']), list(raw_content_df['content_id'])]
shortest = min(lists, key=len)

# Check if all items in the shortest list exist in the other two
others = [lst for lst in lists if lst is not shortest]

if all(item in others[0] and item in others[1] for item in shortest):
    print("All items in the shortest list appear in the other two.")
else:
    print("Not all items from the shortest list are in the others.")

Not all items from the shortest list are in the others.


In [72]:
missing_items = list(set(list(raw_interaction_df['content_id'])) - set(list(news['content_id'])))
print("Items in short_list not in long_list:", missing_items)

Items in short_list not in long_list: ['N51767', 'N20764', 'N2233', 'N50799', 'N63436', 'N14941', 'N8813', 'N7950', 'N45522', 'N32536', 'N4667', 'N29994', 'N19773', 'N28640', 'N15347', 'N7957', 'N39934', 'N51104', 'N18311', 'N16396', 'N5472', 'N47175', 'N15435', 'N11330', 'N62949', 'N12788', 'N21681', 'N24802', 'N12409', 'N56403', 'N63509', 'N22260', 'N37848', 'N11092', 'N11390', 'N23513', 'N31910', 'N48031', 'N33315', 'N29393', 'N58188', 'N33176', 'N14802', 'N46027', 'N21984', 'N55237', 'N57955', 'N11150', 'N55036', 'N52492', 'N10152', 'N31884', 'N52195', 'N37055', 'N2986', 'N31632', 'N54562', 'N50289', 'N32203', 'N28362', 'N29722', 'N28682', 'N25673', 'N16936', 'N53863', 'N13601', 'N27553', 'N34642', 'N2852', 'N35676', 'N34171', 'N14223', 'N43269', 'N30049', 'N44621', 'N512', 'N42120', 'N17194', 'N35159', 'N13022', 'N58251', 'N52103', 'N1106', 'N51470', 'N61838', 'N5183', 'N60762', 'N14555', 'N56080', 'N30598', 'N29379', 'N40875', 'N38951', 'N3451', 'N2735', 'N46624', 'N17807', 'N516

In [73]:
raw_interaction_df = raw_interaction_df[raw_interaction_df['content_id'] != item for]

ValueError: ('Lengths must match to compare', (2431,), (109,))

In [53]:
ranked_recommendations = 'ranked_recommendations/ranked_recommendations.csv'
ranked_recommendations_df = blob_to_df(container_client, ranked_recommendations)
ranked_recommendations_df.isnull().sum().any()
len(ranked_recommendations_df)

2375